# Task 1

## Original Data

In [2]:
import polars as pl

training_csv_path = "/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/train.csv"
temp = pl.scan_csv(training_csv_path)

row_count = temp.select(pl.len()).collect().item()
num_ids = temp.select(pl.col("id").n_unique()).collect().item()
first_stamp = temp.select(pl.col("event_timestamp").min()).collect().item()
last_stamp = temp.select(pl.col("event_timestamp").max()).collect().item()

print(f"Row count: {row_count}")
print(f"Number of unique IDs: {num_ids}")
print(f"First timestamp: {first_stamp}")
print(f"Last timestamp: {last_stamp}")

Row count: 54960961
Number of unique IDs: 1430445
First timestamp: 2020-11-03T03:31:30Z
Last timestamp: 2023-01-23T12:29:56Z


# Task 2

In [3]:
q = temp

# 2. Get the Total Row Count (Raw)
total_count = q.select(pl.len()).collect().item()

# 3. Define the Duplicates 
# We define a duplicate as same ID, same Action (ed_id), and same Timestamp
# We keep the first occurrence
q_unique = q.unique(subset=["id", "customer_id", "event_timestamp"], keep="first")

# 4. Execute the deduplication and get the New Count
df_cleaned = q_unique.collect()
unique_count = df_cleaned.height

# 5. Calculate Statistics
duplicate_count = total_count - unique_count
proportion_duplicates = (duplicate_count / total_count) * 100

# 6. Summary Statistics Evidence
print("--- Deduplication Summary Statistics ---")
print(f"1. Total entries (Raw): {total_count:,}")
print(f"2. Total duplicates found: {duplicate_count:,}")
print(f"3. Proportion of duplicates: {proportion_duplicates:.2f}%")
print(f"4. Rows remaining after cleaning: {unique_count:,}")

--- Deduplication Summary Statistics ---
1. Total entries (Raw): 54,960,961
2. Total duplicates found: 4,515,888
3. Proportion of duplicates: 8.22%
4. Rows remaining after cleaning: 50,445,073


# Task 3

### Convert Time-Date Format

In [ ]:
df_cleaned = df_cleaned.with_columns(
    pl.col("event_timestamp")
    .str.to_datetime(time_zone="UTC") # Tells Polars to expect the Z or +00:00
    .dt.replace_time_zone(None)       # Strips the timezone to make it "naive"
)

SchemaError: invalid series dtype: expected `String`, got `datetime[μs]` for series with name `event_timestamp`

This error occurred in the following expression:
	col("event_timestamp").str.strptime(["raise"])
while evaluating this larger expression:
	col("event_timestamp").str.strptime(["raise"]).dt.replace_time_zone(["raise"])


In [ ]:
# Part 1: Journey Stats:
journey_stats = (
    df_cleaned.group_by("customer_id")
    .agg(
        pl.len().alias("journey_length"), 
        (pl.col("event_timestamp").max() - pl.col("event_timestamp").min()).alias("duration")
    )
    .select(
        pl.col("journey_length").mean().alias("avg_actions"),
        pl.col("journey_length").median().alias("median_actions"),
        pl.col("duration").mean().alias("avg_time"),
        pl.col("duration").median().alias("median_time")
    )
)

print(journey_stats)

shape: (1, 4)
┌─────────────┬────────────────┬─────────────────────────┬─────────────────┐
│ avg_actions ┆ median_actions ┆ avg_time                ┆ median_time     │
│ ---         ┆ ---            ┆ ---                     ┆ ---             │
│ f64         ┆ f64            ┆ duration[μs]            ┆ duration[μs]    │
╞═════════════╪════════════════╪═════════════════════════╪═════════════════╡
│ 36.254357   ┆ 24.0           ┆ 143d 8h 40m 19s 53868µs ┆ 107d 2h 56m 27s │
└─────────────┴────────────────┴─────────────────────────┴─────────────────┘


In [ ]:
# Part 2: Common Actions
action_counts = (
    df_cleaned["ed_id"]
    .value_counts(sort = True)
)

event_def = pl.read_csv("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/Event Definitions.csv")

common_actions = action_counts.join(
    event_def,
    left_on= "ed_id",
    right_on = "event_definition_id",
    how = "left"
).select([
    pl.col("ed_id"),
    pl.col("event_name"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum()).alias("proportion")
])

print(common_actions.head(10))

shape: (10, 4)
┌───────┬──────────────────────────┬──────────┬────────────┐
│ ed_id ┆ event_name               ┆ count    ┆ proportion │
│ ---   ┆ ---                      ┆ ---      ┆ ---        │
│ i64   ┆ str                      ┆ u32      ┆ f64        │
╞═══════╪══════════════════════════╪══════════╪════════════╡
│ 4     ┆ browse_products          ┆ 19421235 ┆ 0.384998   │
│ 5     ┆ view_cart                ┆ 5949607  ┆ 0.117942   │
│ 1     ┆ null                     ┆ 5540997  ┆ 0.109842   │
│ 19    ┆ application_web_view     ┆ 5381039  ┆ 0.106671   │
│ 11    ┆ add_to_cart              ┆ 3694216  ┆ 0.073232   │
│ 6     ┆ begin_checkout           ┆ 2406114  ┆ 0.047698   │
│ 21    ┆ catalog_mail             ┆ 2231987  ┆ 0.044246   │
│ 12    ┆ application_web_approved ┆ 1389756  ┆ 0.02755    │
│ 24    ┆ null                     ┆ 1377718  ┆ 0.027311   │
│ 2     ┆ campaign_click           ┆ 780968   ┆ 0.015482   │
└───────┴──────────────────────────┴──────────┴────────────┘


In [29]:
# Part 3: Time-Between Actions
df_time_delta = df_cleaned.with_columns(
    time_to_next_action = (
        pl.col("event_timestamp")
        .sort()
        .diff()
        .over("customer_id")
    )
)

time_stats = df_time_delta.select([
    pl.col("time_to_next_action").mean().alias("avg_time_between"),
    pl.col("time_to_next_action").median().alias("median_time_between"),
    pl.col("time_to_next_action").max().alias("max_time_between"),
    pl.col("time_to_next_action").min().alias("min_time_between")
])

print(time_stats)

shape: (1, 4)
┌────────────────────────┬─────────────────────┬──────────────────┬──────────────────┐
│ avg_time_between       ┆ median_time_between ┆ max_time_between ┆ min_time_between │
│ ---                    ┆ ---                 ┆ ---              ┆ ---              │
│ duration[μs]           ┆ duration[μs]        ┆ duration[μs]     ┆ duration[μs]     │
╞════════════════════════╪═════════════════════╪══════════════════╪══════════════════╡
│ 4d 1h 35m 44s 352227µs ┆ 1m 37s              ┆ 736d 13h 30m 46s ┆ 0µs              │
└────────────────────────┴─────────────────────┴──────────────────┴──────────────────┘


# Task 4

## Flattened Data

In [4]:
import polars as pl

# 1. Load the Parquet file (Use scan for efficiency)
output_path = "training_new_data.parquet" 
df = pl.read_parquet(output_path)
df.head()

# 2. Check the "Shape" 
#print(f"Dataset Shape: {df.shape}")


id,journey
str,list[struct[2]]
"""-196981038 -1702920673""","[{2021-04-09 06:00:00 UTC,2}, {2021-04-09 22:22:46 UTC,19}, … {2022-02-01 07:21:29 UTC,1}]"
"""-547474919 -1682725744""","[{2021-05-11 04:00:24 UTC,12}, {2021-05-11 04:03:23 UTC,4}, … {2022-02-14 00:00:00 UTC,21}]"
"""1986332210 1321048142""","[{2021-08-05 09:34:35 UTC,12}, {2021-09-07 22:21:54 UTC,1}, … {2022-06-01 07:11:05 UTC,1}]"
"""1315878689 -159556054""","[{2021-10-04 06:00:00 UTC,2}, {2021-10-04 06:00:00 UTC,22}, … {2022-10-10 19:08:46 UTC,4}]"
"""1568974608 1546638717""","[{2021-07-20 06:00:00 UTC,2}, {2021-07-20 10:57:34 UTC,19}, … {2022-05-03 07:17:58 UTC,1}]"


NameError: name 'train' is not defined